<a href="https://colab.research.google.com/github/RajeshworM/IMD_GRID_DATA_EXTRACTION/blob/master/IMDGRD_DailyDownload_TMIN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Colab-ready: download IMD .grd daily files, map grid cells -> districts, produce district-wise CSV
# Run this in Google Colab (one cell). Adjust start_date / end_date and paths as needed.

# 0) NOTE: This will install geopandas, fiona etc. (may take a minute).
!pip install -q geopandas requests tqdm shapely pyproj fiona

import os
import io
import zipfile
import requests
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from tqdm import tqdm

# Geo libraries
import geopandas as gpd
from shapely.geometry import Point

# ------------------ CONFIG ------------------
IMD_BASE = "https://www.imdpune.gov.in/cmpg/Realtimedata/min/"
CTL_URL  = IMD_BASE + "TEMP.CTL"   # official CTL we discovered
# GRD name pattern observed on the site: minDDMMYYYY.grd (lowercase 'min')
GRD_FILENAME_PATTERN = "min{dd}{mm}{yyyy}.grd"

# district boundaries (geoBoundaries raw zip on GitHub) - stable raw URL
# (this is a large-ish file for all ADM2 India; it will be downloaded and extracted)
GEOBOUNDARIES_IND_ADM2_ZIP = "https://github.com/wmgeolab/geoBoundaries/raw/9469f09/releaseData/gbOpen/IND/ADM2/geoBoundaries-IND-ADM2-all.zip"

# Where to store files locally in Colab
WORKDIR = "/content/imdpune_min"
GRD_DIR = os.path.join(WORKDIR, "grds")
GRIDMAP_DIR = os.path.join(WORKDIR, "gridmap")  # for temporary mapping artifacts
GEO_DIR = os.path.join(WORKDIR, "geoboundaries")
OUT_CSV = os.path.join(WORKDIR, "district_daily_min_temp_2025.csv")

os.makedirs(WORKDIR, exist_ok=True)
os.makedirs(GRD_DIR, exist_ok=True)
os.makedirs(GRIDMAP_DIR, exist_ok=True)
os.makedirs(GEO_DIR, exist_ok=True)

# Date range (change if you want a different period)
start_date = datetime(2025, 1, 1)
end_date   = datetime.today()   # latest available according to system clock

# If you want to test quickly, set small_test=True
small_test = False
if small_test:
    end_date = start_date + timedelta(days=2)
# ----------------------------------------------

# ---------- Utility download function ----------
def download_file(url, out_path, stream=True, overwrite=False):
    if os.path.exists(out_path) and not overwrite:
        print("Already have", out_path)
        return out_path
    print("Downloading:", url)
    r = requests.get(url, stream=stream, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"Failed to download {url} (status {r.status_code})")
    with open(out_path, "wb") as fh:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                fh.write(chunk)
    print("Saved ->", out_path)
    return out_path

# ---------- 1) Download and parse CTL ----------
ctl_local = os.path.join(WORKDIR, "TEMP.CTL")
try:
    download_file(CTL_URL, ctl_local)
except Exception as e:
    raise RuntimeError(f"Couldn't download CTL from {CTL_URL}: {e}")

# parse CTL (GrADS-like)
meta = {}
with open(ctl_local, "rt") as f:
    for line in f:
        parts = line.strip().split()
        if not parts:
            continue
        key = parts[0].upper()
        if key == "XDEF":
            meta["ncols"] = int(parts[1])
            # parts example: XDEF 61 LINEAR 67.5 0.5
            meta["x_start"] = float(parts[3])
            meta["x_step"]  = float(parts[4])
        elif key == "YDEF":
            meta["nrows"] = int(parts[1])
            meta["y_start"] = float(parts[3])
            meta["y_step"]  = float(parts[4])
        elif key == "UNDEF":
            meta["undef"] = float(parts[1])

# Basic validation
if not all(k in meta for k in ("ncols","nrows","x_start","x_step","y_start","y_step","undef")):
    raise RuntimeError("CTL parse didn't find expected keys. CTL content:\n" + open(ctl_local).read())

ncols = meta["ncols"]
nrows = meta["nrows"]
nodata_value = meta["undef"]
x_start = meta["x_start"]
y_start = meta["y_start"]
x_step = meta["x_step"]
y_step = meta["y_step"]

print(f"Parsed CTL: nrows={nrows}, ncols={ncols}, x_start={x_start}, y_start={y_start}, x_step={x_step}, y_step={y_step}, nodata={nodata_value}")

# Build grid cell centers (lon, lat)
xs = x_start + np.arange(ncols) * x_step
ys = y_start + np.arange(nrows) * y_step
# Note: we'll assume grid data file rows correspond to Y ascending (first row is y= y_start),
# and reshape as (nrows, ncols) in that order.

grid_points = []
grid_index_to_point = {}  # (r,c) -> (lon, lat)
for r in range(nrows):
    for c in range(ncols):
        lon = xs[c]
        lat = ys[r]
        grid_points.append(Point(lon, lat))
        grid_index_to_point[(r, c)] = (lon, lat)

# Build GeoDataFrame of points
gdf_points = gpd.GeoDataFrame({
    "r": [pt_idx[0] for pt_idx in grid_index_to_point.keys()],
    "c": [pt_idx[1] for pt_idx in grid_index_to_point.keys()],
}, geometry=grid_points, crs="EPSG:4326")

print("Built grid points GeoDataFrame (61x61 -> total points):", len(gdf_points))

# ---------- 2) Download India district boundaries (geoBoundaries ADM2) ----------
geo_zip_local = os.path.join(GEO_DIR, "geoBoundaries-IND-ADM2-all.zip")
try:
    download_file(GEOBOUNDARIES_IND_ADM2_ZIP, geo_zip_local)
except Exception as e:
    raise RuntimeError("Could not download geoBoundaries archive: " + str(e))

# extract zip and find a geojson or shapefile inside
with zipfile.ZipFile(geo_zip_local, "r") as z:
    z.extractall(GEO_DIR)
    members = z.namelist()

# find top-level file containing 'ADM2' and geojson/shp
found_geojson = None
found_shp = None
for m in members:
    if m.lower().endswith(".geojson"):
        found_geojson = os.path.join(GEO_DIR, m)
        break
    if m.lower().endswith(".shp") and "ADM2" in m:
        found_shp = os.path.join(GEO_DIR, m)

if found_geojson:
    print("Found geojson:", found_geojson)
    gdf_admin = gpd.read_file(found_geojson)
elif found_shp:
    print("Found shapefile:", found_shp)
    gdf_admin = gpd.read_file(found_shp)
else:
    # fallback: try reading any geojson shapefile in the extracted folder
    cand = [os.path.join(GEO_DIR, f) for f in members if f.lower().endswith((".shp",".geojson"))]
    if not cand:
        raise RuntimeError("No shapefile/geojson found in extracted geoBoundaries zip.")
    # pick the first candidate
    path0 = os.path.join(GEO_DIR, cand[0]) if not cand[0].startswith(GEO_DIR) else cand[0]
    print("Fallback reading:", path0)
    gdf_admin = gpd.read_file(path0)

print("Loaded admin polygons count:", len(gdf_admin))
# Ensure CRS is WGS84
if gdf_admin.crs is None or gdf_admin.crs.to_string() != "EPSG:4326":
    gdf_admin = gdf_admin.to_crs(epsg=4326)

# Pick a sensible label for district name. Many geoBoundaries files use "shapeName" or "shapeGroup" or "gID".
# We'll try to detect common fields.
possible_name_fields = ["shapeName", "shapeName_en", "NAME_2", "NAME_1", "NAME", "geometry", "shapeID", "NAME"]
district_name_field = None
for f in gdf_admin.columns:
    if f.lower() in ("shapeName".lower(), "shapename", "name_2", "name_1", "name"):
        district_name_field = f
        break
# fallback to first non-geometry field
if district_name_field is None:
    nongeom = [c for c in gdf_admin.columns if c != gdf_admin.geometry.name]
    district_name_field = nongeom[0] if nongeom else None

print("Using district name field:", district_name_field)

# ---------- 3) Spatial join: assign each grid point to a district polygon ----------
# Using spatial join (points within polygons)
print("Performing spatial join: grid points -> district polygons (this may take a few seconds)...")
# Add unique id to polygons
gdf_admin = gdf_admin.reset_index().rename(columns={"index":"poly_index"})
# Perform spatial join
gdf_points = gdf_points.set_geometry("geometry")
gdf_joined = gpd.sjoin(gdf_points, gdf_admin[[district_name_field, "poly_index", "geometry"]], how="left", predicate="within")
# gdf_joined will have columns: r,c, geometry, index_right, district_name_field, poly_index
print("Points after join:", gdf_joined.shape)
# Build a 2D array of district names (or polygon index) matching grid indices (r,c)
grid2district = np.full((nrows, ncols), fill_value=-1, dtype=int)
grid2district_name = np.full((nrows, ncols), fill_value=None, dtype=object)
for idx, row in gdf_joined.iterrows():
    r = int(row["r"])
    c = int(row["c"])
    if pd.isna(row["poly_index"]):
        grid2district[r, c] = -1
        grid2district_name[r, c] = None
    else:
        pid = int(row["poly_index"])
        grid2district[r, c] = pid
        if district_name_field:
            grid2district_name[r, c] = str(row[district_name_field])

# Map polygon index -> district name (unique)
polyindex_to_name = {}
for pid in np.unique(grid2district):
    if pid == -1:
        continue
    # find the polygon row
    row = gdf_admin.loc[gdf_admin["poly_index"] == pid]
    if len(row) > 0:
        polyindex_to_name[pid] = str(row.iloc[0][district_name_field])
    else:
        polyindex_to_name[pid] = f"poly_{pid}"

print("Unique districts found on the grid:", len(polyindex_to_name))
# Save mapping (optional)
np.save(os.path.join(GRIDMAP_DIR, "grid2district_index.npy"), grid2district)
# Also save names table
pd.DataFrame.from_dict(polyindex_to_name, orient="index", columns=["district_name"]).to_csv(os.path.join(GRIDMAP_DIR, "polyindex_to_name.csv"))

# If some grid cells had no polygon (e.g., ocean), that's okay; they will be ignored during aggregation.

# ---------- 4) Download all .grd files for the date range ----------
print("\nDownloading daily .grd files to:", GRD_DIR)
date = start_date
downloaded_files = []
pbar = tqdm(total=(end_date - start_date).days + 1)
while date <= end_date:
    fname = GRD_FILENAME_PATTERN.format(dd=f"{date.day:02d}", mm=f"{date.month:02d}", yyyy=f"{date.year}")
    url = IMD_BASE + fname
    local_path = os.path.join(GRD_DIR, fname)
    try:
        if not os.path.exists(local_path):
            r = requests.get(url, stream=True, timeout=30)
            if r.status_code == 200:
                with open(local_path, "wb") as fh:
                    for chunk in r.iter_content(chunk_size=8192):
                        if chunk:
                            fh.write(chunk)
                downloaded_files.append(local_path)
            else:
                # not found - print but continue
                print("Not available:", url, "(status", r.status_code, ")")
            # small sleep avoided to speed up; if rate-limited, you can add time.sleep(0.5)
        else:
            downloaded_files.append(local_path)
    except Exception as e:
        print("Error while downloading", url, ":", e)
    date += timedelta(days=1)
    pbar.update(1)
pbar.close()
print("Downloaded/Found GRD files:", len(downloaded_files))

if len(downloaded_files) == 0:
    raise RuntimeError("No .grd files downloaded. Check date range and IMD site availability.")

# ---------- 5) Robust .grd file reader (tries endianness / orientation) ----------
def read_grid_best_guess(path, nrows=nrows, ncols=ncols, nodata=nodata_value):
    # Try little-endian float32 first (common)
    candidates = []
    try:
        arr_le = np.fromfile(path, dtype="<f4")
        if arr_le.size == nrows * ncols:
            arr_le = arr_le.reshape((nrows, ncols))
            candidates.append(("le", arr_le))
    except Exception:
        pass
    try:
        arr_be = np.fromfile(path, dtype=">f4")
        if arr_be.size == nrows * ncols:
            arr_be = arr_be.reshape((nrows, ncols))
            candidates.append(("be", arr_be))
    except Exception:
        pass
    # Also try C-order vs Fortran-order swap (transpose)
    # If both exist, pick the one with most plausible temperature values
    def plausibility_score(a):
        # fraction of values that are in a reasonable temperature range (-90,70)
        a_flat = a.flatten()
        valid = a_flat[(~np.isnan(a_flat)) & (a_flat != nodata)]
        if valid.size == 0:
            return 0.0
        frac = np.sum((valid > -90.0) & (valid < 70.0)) / valid.size
        return frac

    if not candidates:
        # fallback: raw as float32 default
        raw = np.fromfile(path, dtype=np.float32)
        if raw.size != nrows*ncols:
            raise RuntimeError(f"Grid file {path} has unexpected size {raw.size}, expected {nrows*ncols}")
        arr = raw.reshape((nrows, ncols))
        return arr
    # evaluate scores
    scored = []
    for tag, arr in candidates:
        scored.append((plausibility_score(arr), tag, arr))
        # also try transpose
        scored.append((plausibility_score(arr.T), tag + "_T", arr.T))
    # pick best
    best = max(scored, key=lambda x: x[0])
    if best[0] < 0.2:
        # if very low plausibility, still return first candidate but warn
        print("Warning: low plausibility for", path, "best score", best[0])
    return best[2]

# ---------- 6) Process each .grd -> district means and combine ----------
print("\nProcessing .grd files and aggregating to districts...")

# Get list of district poly indexes we found
district_poly_indexes = sorted(list(polyindex_to_name.keys()))
# We'll produce a dataframe of rows: date + columns for each district_name
records = []
for local_path in tqdm(sorted(downloaded_files)):
    fname = os.path.basename(local_path)
    # guess date from filename pattern minDDMMYYYY.grd
    try:
        dd = int(fname[3:5]); mm = int(fname[5:7]); yyyy = int(fname[7:11])
        dt = datetime(yyyy, mm, dd)
        dt_str = dt.strftime("%Y-%m-%d")
    except Exception:
        dt_str = fname
    try:
        arr = read_grid_best_guess(local_path, nrows=nrows, ncols=ncols, nodata=nodata_value)
    except Exception as e:
        print("Skipping file (read error):", local_path, ":", e)
        continue

    # For each district polygon index, average the grid cells that map to it
    rec = {"date": dt_str}
    for pid in district_poly_indexes:
        mask = (grid2district == pid)
        if not np.any(mask):
            rec[polyindex_to_name[pid]] = np.nan
            continue
        vals = arr[mask]
        vals = vals[(~np.isnan(vals)) & (vals != nodata_value)]
        rec[polyindex_to_name[pid]] = float(np.nan) if len(vals) == 0 else float(np.mean(vals))
    records.append(rec)

# combined DataFrame
if len(records) == 0:
    raise RuntimeError("No valid records produced from .grd files (all failed).")

df = pd.DataFrame(records)
# ensure date first and sorted
cols = ['date'] + [c for c in df.columns if c != 'date']
df = df[cols].sort_values("date").reset_index(drop=True)
df.to_csv(OUT_CSV, index=False)
print("\nSaved combined CSV:", OUT_CSV)
print("CSV shape:", df.shape)
from google.colab import files

# Provide download link for the CSV
files.download(OUT_CSV)


Already have /content/imdpune_min/TEMP.CTL
Parsed CTL: nrows=61, ncols=61, x_start=67.5, y_start=7.5, x_step=0.5, y_step=0.5, nodata=99.9
Built grid points GeoDataFrame (61x61 -> total points): 3721
Already have /content/imdpune_min/geoboundaries/geoBoundaries-IND-ADM2-all.zip
Found geojson: /content/imdpune_min/geoboundaries/geoBoundaries-IND-ADM2.geojson
Loaded admin polygons count: 735
Using district name field: shapeName
Performing spatial join: grid points -> district polygons (this may take a few seconds)...
Points after join: (3721, 6)
Unique districts found on the grid: 572



100%|██████████| 269/269 [00:00<00:00, 53409.13it/s]


Downloaded/Found GRD files: 269

Processing .grd files and aggregating to districts...


100%|██████████| 269/269 [00:04<00:00, 55.79it/s]



Saved combined CSV: /content/imdpune_min/district_daily_min_temp_2025.csv
CSV shape: (269, 567)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>